# inplace-param-update — ex2: hand-rolled SGD over a parameter list using .data.sub_

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `inplace-param-update`. Running the final beacon cell reports progress against the `PyTorch: In-place param update` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: In-place param update` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`inplace-param-update`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "inplace-param-update"
DD_SUBTOPIC = "PyTorch: In-place param update"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## in-place parameter update — quick refresher

A hand-rolled optimizer step has to mutate the existing parameter tensor — NOT rebind a Python name. Two equivalent in-place patterns:

- `param.data -= lr * grad` — works on `nn.Parameter` (the `.data` alias keeps autograd-tracked Parameter wrapper intact).
- `param.data.sub_(grad, alpha=lr)` — same effect, slightly faster, matches what `torch.optim.SGD.step` actually does.

**Why `.data` and not just `param -= lr * grad`.** On a leaf Parameter that requires grad, you can't run in-place ops in a way autograd would track without `.data` (or a `with torch.no_grad()` block). Both bypass autograd; `.data` is the older idiom, `no_grad` is what new code uses.

**Storage identity = registered-buffer link.** If you rebind a Parameter (`param = param - lr * g`), the module's `parameters()` list still points at the OLD tensor — the new one is orphaned. Optimizers + checkpoint save/load all break.

### Exercise 2 — hand-rolled SGD over a parameter list using .data.sub_

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `param.data.sub_(grad, alpha=lr)` across a list of `nn.Parameter` objects so that every parameter's storage pointer is preserved after the step.
> Keywords: sgd, parameter-list, data-sub_, storage-identity
> ```

**KCs targeted:** `inplace-update-mutates-storage`, `param-data-bypass-autograd`

Implement `ex2_sgd_step(params, grads, lr)`. The minimal hand-rolled SGD update over an arbitrary parameter list:

1. `params` is a list of `nn.Parameter` (each requires_grad=True).
2. `grads` is a list of `Tensor` of matching shapes — pretend these came from a backward pass (we pass them in directly to keep the test deterministic).
3. `lr` is a Python float.
4. For each `(p, g)` pair: do `p.data.sub_(g, alpha=lr)` (in-place).
5. Return the list `params` itself (same Python list, same elements).

**The test will record each parameter's `data_ptr()` BEFORE the call and assert they are identical AFTER.** A rebinding implementation (e.g. `p = p - lr * g`) will fail because the storage changes.

You are NOT inside a `torch.no_grad()` block in this function — `.data` is doing the autograd-bypass for you.

In [ ]:
import torch.nn as nn

def ex2_sgd_step(params: list, grads: list, lr: float) -> list:
    for p, g in zip(params, grads):
        p.data.sub_(g, alpha=lr)
    return params


<details><summary>Solution</summary>

```python
import torch.nn as nn

def ex2_sgd_step(params: list, grads: list, lr: float) -> list:
    for p, g in zip(params, grads):
        p.data.sub_(g, alpha=lr)
    return params
```

**`sub_(g, alpha=lr)` not `data -= lr * g`.** Both work, but `sub_` fuses the scale+subtract into one kernel call and matches what `torch.optim.SGD` does internally (no momentum/weight-decay here).

**Never `p.data = ...`.** Assigning a new tensor to `p.data` keeps the Parameter wrapper but swaps its underlying storage — anything holding a reference to the OLD storage (saved activations, optimizer state, hooks) is now stale. `.sub_` mutates the existing storage in place.

**Why `.data` not `with no_grad()`.** On a leaf Parameter that requires grad, in-place ops on the Parameter itself raise. `.data` is the untracked view; `with no_grad():` is the modern equivalent. ARENA training loops historically use `.data`, so we drilled that.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()